# Create User and Client Profiles

This notebook demonstrates how to create user and client profiles using the modern Person/Actor architecture with Pydantic models, bidirectional User-Client assignment, and YAML persistence via HydraConfigManager.

**Test case**: Hillcrest Children & Family Center as a Client of Masai Interactive, managed by Dheeraj Chand (Siege Analytics).

In [ ]:
# Import required modules — modern Person/Actor architecture
from pathlib import Path
from siege_utilities.config import User, Client, BrandingConfig, ReportPreferences
from siege_utilities.config import HydraConfigManager
import siege_utilities as su

# Initialize logging
su.configure_shared_logging(level="INFO")

# --- Path configuration ---
# Resolve and ensure the default download directory exists.
from siege_utilities.config.user_config import get_user_config

DATA_DIR = Path(get_user_config().get_download_directory())
DATA_DIR.mkdir(parents=True, exist_ok=True)

su.log_info(f"Data directory: {DATA_DIR}  (exists={DATA_DIR.exists()})")
su.log_info("Starting profile creation demo")

## 1. Creating a User Profile

Let's create a comprehensive user profile with all available options:

In [ ]:
import os
from pathlib import Path

# Define user identity variables up front
username = "dheerajchand"
full_name = "Dheeraj Chand"
email = "dheeraj@siegeanalytics.com"

# Create a comprehensive user profile using the modern User model
user = User(
    # Core Identity (from Person base)
    person_id=username,
    name=full_name,
    email=email,
    
    # User-specific fields
    username=username,
    github_login=username,
    role="admin",
    
    # Preferences (cross-platform path)
    preferred_download_directory=str(Path.home() / "Downloads" / "siege_utilities"),
    default_output_format="pdf",
    preferred_map_style="carto-positron",
    default_color_scheme="viridis",
    
    # Technical Preferences
    default_dpi=300,
    default_figure_size=(12, 8),
    enable_logging=True,
    log_level="INFO",
    
    # API Keys (optional — will be populated from 1Password)
    google_analytics_key="",
    facebook_business_key="",
    census_api_key="",
    
    # Database Preferences
    default_database="postgresql",
    postgresql_connection="",
    duckdb_path=""
)

su.log_info("User Created Successfully!")
su.log_info(f"Person ID: {user.person_id}")
su.log_info(f"Name: {user.name}")
su.log_info(f"Email: {user.email}")
su.log_info(f"Username: {user.username}")
su.log_info(f"Role: {user.role}")
su.log_info(f"Download Directory: {user.preferred_download_directory}")
su.log_info(f"Output Format: {user.default_output_format}")
su.log_info(f"Color Scheme: {user.default_color_scheme}")
su.log_info(f"Map Style: {user.preferred_map_style}")
su.log_info(f"DPI: {user.default_dpi}")
su.log_info(f"Figure Size: {user.default_figure_size}")

## 2. Creating Branding Configuration

Now let's create a branding configuration for Hillcrest Children & Family Center:

In [ ]:
# Create branding configuration for Hillcrest with hex color validation
hillcrest_branding = BrandingConfig(
    # Colors (must be valid hex codes) — Hillcrest brand palette
    primary_color="#1a4d2e",      # Forest green
    secondary_color="#4a7c59",    # Sage green
    accent_color="#f4a261",       # Warm orange
    text_color="#2d3436",         # Dark gray
    background_color="#ffffff",   # White
    
    # Typography
    primary_font="Helvetica",
    secondary_font="Georgia",
    
    # Logo (optional — will be set when logo file is available)
    logo_path=None,
    logo_width=None,
    logo_height=None,
    
    # Layout dimensions
    header_height=50,
    footer_height=25,
    margin_top=25,
    margin_bottom=25,
    
    # Chart styling
    chart_color_palette="viridis"
)

su.log_info("Hillcrest Branding Configuration Created!")
su.log_info(f"Primary Color: {hillcrest_branding.primary_color}")
su.log_info(f"Secondary Color: {hillcrest_branding.secondary_color}")
su.log_info(f"Accent Color: {hillcrest_branding.accent_color}")
su.log_info(f"Primary Font: {hillcrest_branding.primary_font}")
su.log_info(f"Secondary Font: {hillcrest_branding.secondary_font}")
su.log_info(f"Header Height: {hillcrest_branding.header_height}px")
su.log_info(f"Footer Height: {hillcrest_branding.footer_height}px")

## 3. Creating a Client Profile

Now let's create Hillcrest Children & Family Center as a Client of Masai Interactive, with Dheeraj as the assigned user:

In [ ]:
# Create Hillcrest as a Client using the modern Client model
client = Client(
    # Core Identity (from Person base)
    person_id="hillcrest_client",
    name="Hillcrest Children & Family Center",
    email="info@hillcrestchildren.org",
    website="https://www.hillcrestchildren.org",
    address="Washington, DC",
    
    # Client-specific fields
    client_code="HILL",          # Must be uppercase, 2-10 characters
    industry="Nonprofit - Children & Family Services",
    project_count=1,
    client_status="active",      # active, inactive, or archived
    
    # Organization context — Hillcrest is a client of Masai Interactive
    organizations=["masai_interactive"],
    primary_organization="masai_interactive",
    
    # Client-specific configurations
    branding_config=hillcrest_branding,
    report_preferences=ReportPreferences()  # Use defaults
)

# Demonstrate User-Client bidirectional assignment
client.assign_user(user.person_id)
client.set_primary_user(user.person_id)
user.assign_client(client.client_code)
user.set_primary_client(client.client_code)

su.log_info("Client Created Successfully!")
su.log_info(f"Person ID: {client.person_id}")
su.log_info(f"Name: {client.name}")
su.log_info(f"Client Code: {client.client_code}")
su.log_info(f"Industry: {client.industry}")
su.log_info(f"Project Count: {client.project_count}")
su.log_info(f"Status: {client.client_status}")
su.log_info(f"Email: {client.email}")
su.log_info(f"Website: {client.website}")
su.log_info(f"Address: {client.address}")
su.log_info(f"Organization: {client.primary_organization}")
su.log_info(f"Brand Primary Color: {client.branding_config.primary_color}")
su.log_info(f"Brand Primary Font: {client.branding_config.primary_font}")
su.log_info(f"Assigned Users: {client.get_assigned_users()}")
su.log_info(f"Primary User: {client.primary_user}")
su.log_info(f"User's Assigned Clients: {user.get_assigned_clients()}")
su.log_info(f"User's Primary Client: {user.primary_client}")

## 4. Validation Examples

Let's see how the validation system works with some examples:

In [ ]:
# Test email validation on modern User model
su.log_info("Testing Email Validation:")
try:
    valid_email_user = User(
        person_id="test-user", name="Test User", email="valid@example.com", username="testuser"
    )
    su.log_info("   Valid email: valid@example.com")
except ValueError as e:
    su.log_error(f"   Error: {e}")

try:
    invalid_email_user = User(
        person_id="test-user2", name="Test User", email="invalid-email", username="testuser2"
    )
    su.log_info("   Invalid email handled gracefully")
except ValueError as e:
    su.log_error(f"   Validation error: {e}")

# Test color validation
su.log_info("Testing Color Validation:")
try:
    valid_colors = BrandingConfig(
        primary_color="#FF0000",
        secondary_color="#00FF00",
        accent_color="#0000FF",
        text_color="#000000",
        background_color="#FFFFFF",
        primary_font="Arial",
        secondary_font="Arial"
    )
    su.log_info("   Valid hex colors accepted")
except ValueError as e:
    su.log_error(f"   Error: {e}")

try:
    invalid_colors = BrandingConfig(
        primary_color="red",  # Invalid hex format
        secondary_color="#00FF00",
        accent_color="#0000FF",
        text_color="#000000",
        background_color="#FFFFFF",
        primary_font="Arial",
        secondary_font="Arial"
    )
except ValueError as e:
    su.log_error(f"   Invalid color format correctly rejected: {e}")

## 5. Loading Configurations with HydraConfigManager

Let's see how to load configurations using the HydraConfigManager:

In [ ]:
# Save and load with modern HydraConfigManager methods
import tempfile
from pathlib import Path

with HydraConfigManager() as manager:
    # Use a temp directory for this demo
    tmp_dir = Path(tempfile.mkdtemp())
    
    # Save the user we created above
    saved = manager.save_user(user, profiles_dir=tmp_dir / "users")
    su.log_info(f"User saved: {saved}")
    
    # Save the client we created above
    saved = manager.save_client(client, profiles_dir=tmp_dir / "clients")
    su.log_info(f"Client saved: {saved}")
    
    # Load them back
    loaded_user = manager.load_user("dheerajchand", profiles_dir=tmp_dir / "users")
    if loaded_user:
        su.log_info("Loaded User:")
        su.log_info(f"   Person ID: {loaded_user.person_id}")
        su.log_info(f"   Name: {loaded_user.name}")
        su.log_info(f"   Email: {loaded_user.email}")
        su.log_info(f"   Username: {loaded_user.username}")
        su.log_info(f"   Role: {loaded_user.role}")
        su.log_info(f"   Assigned Clients: {loaded_user.get_assigned_clients()}")
        su.log_info(f"   Primary Client: {loaded_user.primary_client}")
    
    loaded_client = manager.load_client("HILL", profiles_dir=tmp_dir / "clients")
    if loaded_client:
        su.log_info("Loaded Client:")
        su.log_info(f"   Person ID: {loaded_client.person_id}")
        su.log_info(f"   Name: {loaded_client.name}")
        su.log_info(f"   Client Code: {loaded_client.client_code}")
        su.log_info(f"   Industry: {loaded_client.industry}")
        su.log_info(f"   Organization: {loaded_client.primary_organization}")
        su.log_info(f"   Brand Primary Color: {loaded_client.branding_config.primary_color}")
        su.log_info(f"   Assigned Users: {loaded_client.get_assigned_users()}")
        su.log_info(f"   Primary User: {loaded_client.primary_user}")
    
    # Clean up temp directory
    import shutil
    shutil.rmtree(tmp_dir)

## Summary

This notebook demonstrates how to create and manage user and client profiles using the modern Person/Actor architecture.

### What We Covered:
1. **User Creation**: Using the `User` model with Person base fields plus user-specific preferences
2. **Branding Configuration**: Color schemes, fonts, and layout settings with typed `BrandingConfig` validation
3. **Client Creation**: Using the `Client` model with Person base fields plus client-specific configuration
4. **User-Client Assignment**: Bidirectional assignment with `assign_client`/`assign_user` and `set_primary_client`/`set_primary_user`
5. **Validation Examples**: How the system catches and handles invalid data
6. **YAML Persistence**: Using `HydraConfigManager.save_user()`/`load_user()` and `save_client()`/`load_client()`

### Next Steps:
- **Run this notebook** to see the profiles in action
- **Modify the examples** with your own data
- **See NB03** for the full Person/Actor architecture documentation
- **Explore the migration system** for existing configurations

### Additional Resources:
- [Complete Configuration Documentation](../docs/HYDRA_PYDANTIC_CONFIGURATION.md)
- [User Guide](../docs/USER_GUIDE.md)
- [Configuration System Demo](./01_Configuration_System_Demo.ipynb)
- [Person/Actor Architecture](./03_Person_Actor_Architecture.ipynb)